## Agentic RAG

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.06.23 </div>
<div style="text-align: right"> last update : 2026.06.23 </div>

개정 이력  
- `2026.06.23` : 노트북 최초 작성

### Agentic RAG란?

`2.2_rag.ipynb` 에서 만든 RAG는 **고정 파이프라인** 이었습니다. 질문이 무엇이든 흐름이 항상 똑같았죠.

> 질문 → **항상 검색** → 프롬프트 → 답변

이 방식은 단순하고 예측 가능하지만, 두 가지 아쉬움이 있습니다.

1. **안전·법령과 무관한 질문**(예: "오늘 날씨 어때?")에도 굳이 문서를 검색한다.
2. 검색 말고 **다른 도구**(날짜·날씨 등)가 필요한 질문은 처리할 수 없다.

**Agentic RAG** 는 검색을 *항상 하는 단계* 가 아니라 **에이전트가 골라 쓰는 하나의 도구(tool)** 로 바꿉니다. 에이전트는 질문을 보고 *"이건 법령 질문이니 검색 도구를 쓰자"*, *"이건 날씨 질문이니 날씨 도구를 쓰자"*, *"이건 인사니까 그냥 답하자"* 를 **스스로 판단** 합니다.

> 질문 → **[에이전트 판단]** → 필요한 도구 호출 → 결과 관찰 → 답변

이 노트북에서는 `1.3` 의 ReAct 에이전트(`create_agent`) 위에, `2.2` 의 retriever를 **검색 도구** 로 얹어 *질문에 따라 도구를 골라 쓰는* 에이전트를 만듭니다. 특히 **도구 설명(독스트링)과 시스템 프롬프트** 가 도구 선택 품질을 어떻게 좌우하는지에 집중합니다.

### 0. 환경 설정

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
print("LLM       :", os.getenv("ANTHROPIC_MODEL"))
print("Embedding :", os.getenv("VOYAGE_EMBEDDING_MODEL"))

LLM       : claude-haiku-4-5
Embedding : voyage-4-lite


### 1. 벡터 DB 로드 → retriever

`2.1` 에서 저장하고 `2.2` 에서 불러왔던 FAISS 인덱스를 **똑같이** 불러옵니다. (임베딩 모델은 저장할 때와 같은 `voyage-4-lite` 를 써야 합니다.)

여기까지는 `2.2` 와 동일합니다. 다만 이번에는 이 `retriever` 를 직접 호출하지 않고, 다음 단계에서 **에이전트가 쓸 수 있는 도구로 감쌀** 것입니다.

In [3]:
import time

from langchain_voyageai import VoyageAIEmbeddings


class RateLimitedVoyageEmbeddings(VoyageAIEmbeddings):
    """무료 한도(3 RPM / 10K TPM) 초과 시 잠시 대기 후 자동 재시도하는 임베딩.

    RateLimitError 가 나면 25초 기다렸다가 다시 시도하므로,
    빌드·검색 어디서 호출되든 한도에 걸려도 알아서 통과한다.
    (근본 해결은 결제수단 등록 — 무료 토큰은 그대로 적용된다.)
    """

    def _with_retry(self, fn, arg):
        for attempt in range(6):
            try:
                return fn(arg)
            except Exception as e:
                if "RateLimit" not in type(e).__name__:
                    raise
                print(f"  [rate limit] 25초 대기 후 재시도 ({attempt + 1}/6)")
                time.sleep(25)
        raise RuntimeError("재시도 초과 - 잠시 후 다시 실행하거나 결제수단을 등록하세요.")

    def embed_documents(self, texts):
        return self._with_retry(super().embed_documents, texts)

    def embed_query(self, text):
        return self._with_retry(super().embed_query, text)

from langchain_community.vectorstores import FAISS

embeddings = RateLimitedVoyageEmbeddings(model=os.getenv("VOYAGE_EMBEDDING_MODEL"))

vectorstore = FAISS.load_local(
    "../data/vector_db/osh_act_faiss",
    embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("벡터 DB 로드 완료")

벡터 DB 로드 완료


### 2. 도구(Tool) 만들기 ⭐

에이전트가 골라 쓸 도구 세 개를 준비합니다. `@tool` 데코레이터를 붙이고, **독스트링(설명문)** 에 *"언제 이 도구를 써야 하는지"* 를 분명히 적는 것이 핵심입니다. 에이전트는 이 설명만 읽고 도구를 고르기 때문입니다.

#### 2-1. 검색 도구 — RAG를 '도구'로 감싸기

`2.2` 에서는 retriever를 함수 안에서 직접 호출했습니다. 여기서는 retriever를 **`@tool` 로 감싸** 에이전트가 *필요할 때만* 부르게 만듭니다.

독스트링에 **"산업안전보건법·안전·보건 관련 질문일 때 쓰라"** 고 명시하는 것이 중요합니다. 이 한 줄이 에이전트의 라우팅(도구 선택) 판단 근거가 됩니다. 또한 검색 결과에 **페이지 번호** 를 붙여 돌려주어, 에이전트가 답변에 출처를 함께 밝힐 수 있게 합니다.

In [4]:
from langchain.tools import tool


@tool
def search_osh_act(query: str) -> str:
    """산업안전보건법 문서에서 질문과 관련된 조항을 검색한다.

    산업안전·보건, 사업주/근로자의 의무, 중대재해, 도급, 작업중지,
    안전보건진단 등 '법령·안전' 관련 질문일 때 사용한다.
    검색된 조항 본문을 (페이지 번호와 함께) 돌려준다.
    """
    docs = retriever.invoke(query)
    return "\n\n".join(
        f"(p.{doc.metadata.get('page')}) {doc.page_content}" for doc in docs
    )

#### 2-2. 날짜·날씨 도구 (1.3 ReAct 실습에서 재사용)

`1.3` 에서 만든 `get_today` 와 `get_weather` 를 그대로 가져옵니다. 검색 도구와 **나란히** 두면, *법령 질문이 아닌* 질문이 들어왔을 때 에이전트가 검색 대신 이 도구들을 고르는 모습을 확인할 수 있습니다. (날씨는 실제 API 대신 **목업** 으로 서울 = 맑음 고정)

In [5]:
from datetime import date


@tool
def get_today() -> str:
    """오늘 날짜를 'YYYY년 MM월 DD일' 형식으로 알려준다."""
    return date.today().strftime("%Y년 %m월 %d일")


@tool
def get_weather(city: str) -> str:
    """도시 이름을 입력받아 오늘 그 도시의 날씨를 알려준다. (실습용 목업 데이터)"""
    weather_db = {
        "서울": "맑음 (최고 24도 / 최저 14도)",
    }
    return weather_db.get(city, f"'{city}'의 날씨 정보가 없습니다.")

### 3. 시스템 프롬프트 설계 ⭐

Agentic RAG에서 프롬프트는 **"어떤 상황에 어떤 도구를 쓰고, 결과를 어떻게 다룰지"에 대한 규칙서** 입니다. 좋은 프롬프트는 다음을 담습니다.

- **페르소나 / 도구 카탈로그** : 어떤 역할이며, 어떤 도구가 있고 각각 언제 쓰는지
- **라우팅 규칙** : 법령 질문 → 검색, 날짜·날씨 → 해당 도구, 잡담 → 도구 없이 답변
- **그라운딩 규칙** : `2.2` 의 핵심 교훈을 그대로 이식 — 검색 결과로 답할 땐 *문서 내용만* 근거로 하고, 근거가 없으면 솔직히 "찾을 수 없다"고 답한다. (환각 방지의 핵심 장치)
- **출력 형식** : 한국어로, 가능하면 **근거 조항·페이지** 를 함께 제시

도구를 쥐여주는 것만으로는 부족합니다. *언제 쓰고, 결과를 어떻게 신뢰할지* 를 프롬프트로 분명히 알려줘야 라우팅과 답변 품질이 안정됩니다.

In [6]:
agentic_rag_prompt = """너는 '산업안전보건법'과 생활 정보를 함께 도와주는 한국어 어시스턴트야.

[사용할 수 있는 도구]
- search_osh_act : 산업안전보건법 문서에서 관련 조항을 검색한다.
    → 산업안전·보건, 사업주/근로자 의무, 중대재해, 도급, 작업중지, 안전보건진단 등
      '법령·안전' 관련 질문에 사용한다.
- get_today : 오늘 날짜를 알려준다.
- get_weather : 특정 도시의 오늘 날씨를 알려준다.

[행동 규칙]
1. 먼저 질문에 어떤 정보가 필요한지, 어떤 도구를 써야 할지 생각한다.
2. 산업안전보건법·안전·보건 관련 질문이면 반드시 search_osh_act 로 근거를 찾는다.
   너의 사전지식이나 추측으로 답하지 않는다.
3. search_osh_act 결과로 답할 때는 검색된 문서 내용만 근거로 삼는다.
   문서에서 근거를 찾을 수 없으면 "제공된 문서에서 해당 내용을 찾을 수 없습니다." 라고 솔직히 답한다.
4. 날짜·날씨는 추측하지 말고 반드시 도구를 호출해 확인한다.
5. 인사·잡담처럼 도구가 필요 없는 질문은 도구를 호출하지 말고 바로 답한다.
6. 답변은 한국어로, 핵심을 먼저 간결하게 말한다.
   법령 답변은 가능하면 근거 조항(예: 제5조)이나 페이지를 함께 밝힌다.
"""

print(agentic_rag_prompt)

너는 '산업안전보건법'과 생활 정보를 함께 도와주는 한국어 어시스턴트야.

[사용할 수 있는 도구]
- search_osh_act : 산업안전보건법 문서에서 관련 조항을 검색한다.
    → 산업안전·보건, 사업주/근로자 의무, 중대재해, 도급, 작업중지, 안전보건진단 등
      '법령·안전' 관련 질문에 사용한다.
- get_today : 오늘 날짜를 알려준다.
- get_weather : 특정 도시의 오늘 날씨를 알려준다.

[행동 규칙]
1. 먼저 질문에 어떤 정보가 필요한지, 어떤 도구를 써야 할지 생각한다.
2. 산업안전보건법·안전·보건 관련 질문이면 반드시 search_osh_act 로 근거를 찾는다.
   너의 사전지식이나 추측으로 답하지 않는다.
3. search_osh_act 결과로 답할 때는 검색된 문서 내용만 근거로 삼는다.
   문서에서 근거를 찾을 수 없으면 "제공된 문서에서 해당 내용을 찾을 수 없습니다." 라고 솔직히 답한다.
4. 날짜·날씨는 추측하지 말고 반드시 도구를 호출해 확인한다.
5. 인사·잡담처럼 도구가 필요 없는 질문은 도구를 호출하지 말고 바로 답한다.
6. 답변은 한국어로, 핵심을 먼저 간결하게 말한다.
   법령 답변은 가능하면 근거 조항(예: 제5조)이나 페이지를 함께 밝힌다.



### 4. 에이전트 만들기

`1.3` 과 똑같이 `create_agent` 한 줄로 만듭니다. 달라진 점은 **도구 목록에 검색 도구(`search_osh_act`)가 추가** 되었다는 것뿐입니다. 이것만으로 "필요할 때 문서를 검색하는" Agentic RAG 에이전트가 완성됩니다.

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[search_osh_act, get_today, get_weather],
    system_prompt=agentic_rag_prompt,
)

### 5. 추론·도구 호출 과정 출력하기

에이전트가 주고받은 **모든 메시지** 를 순서대로 출력하는 헬퍼를 `1.3` 에서 그대로 가져옵니다. 이렇게 하면 *'어떤 도구를 골랐는지'* 가 그대로 보여, 질문에 따라 라우팅이 어떻게 달라지는지 눈으로 확인할 수 있습니다.

- `[모델의 생각 / 답변]` : 모델이 작성한 추론 또는 최종 답
- `[도구 호출]` : 모델이 **어떤 도구를 어떤 인자로** 호출했는지
- `[도구 결과]` : 도구가 돌려준 값

In [8]:
from langchain.messages import HumanMessage, AIMessage, ToolMessage


def _to_text(content) -> str:
    """메시지 content가 문자열이든 블록 리스트든 텍스트만 뽑아낸다."""
    if isinstance(content, str):
        return content
    chunks = []
    for block in content or []:
        if isinstance(block, str):
            chunks.append(block)
        elif isinstance(block, dict) and block.get("type") == "text":
            chunks.append(block.get("text", ""))
    return "".join(chunks)


def run_react(agent, question):
    """에이전트를 실행하고 추론·도구 호출·도구 결과·최종 답변을 순서대로 출력한다."""
    result = agent.invoke({"messages": [HumanMessage(content=question)]})

    for msg in result["messages"]:
        if isinstance(msg, HumanMessage):
            print("[사용자 질문]")
            print(msg.content)
        elif isinstance(msg, AIMessage):
            text = _to_text(msg.content)
            if text.strip():
                print("[모델의 생각 / 답변]")
                print(text)
            for call in msg.tool_calls:
                print(f"[도구 호출] {call['name']} (인자: {call['args']})")
        elif isinstance(msg, ToolMessage):
            print(f"[도구 결과] {msg.name} → {msg.content[:200]}")
        print("-" * 60)

    return result

### 6. 시나리오 비교 — 에이전트는 질문에 따라 도구를 고른다

이제 성격이 다른 네 가지 질문을 던져, 에이전트가 **질문마다 다른 선택** 을 하는지 관찰합니다. 같은 에이전트인데도 호출하는 도구(혹은 도구를 안 쓰는 것)가 달라지는 점에 주목하세요.

#### 시나리오 A — 안전·법령 질문 → 검색 도구 호출

산업안전보건법과 관련된 질문입니다. 에이전트가 `search_osh_act` 를 **스스로 호출** 해 조항을 찾고, 그 내용을 근거로 답하는지 확인합니다.

In [9]:
_ = run_react(agent, "근로자의 의무는 무엇인가요?")

[사용자 질문]
근로자의 의무는 무엇인가요?
------------------------------------------------------------
[모델의 생각 / 답변]
산업안전보건법에서 정한 근로자의 의무를 검색하겠습니다.
[도구 호출] search_osh_act (인자: {'query': '근로자 의무'})
------------------------------------------------------------
[도구 결과] search_osh_act → (p.12) 작업장소에서 대피시키는 등 안전 및 보건에 관하여 필요한 조치를 하여야 한다.
 
제52조(근로자의 작업중지) ① 근로자는 산업재해가 발생할 급박한 위험이 있는 경우에는 작업을 중지하고 대피할 수
있다.
② 제1항에 따라 작업을 중지하고 대피한 근로자는 지체 없이 그 사실을 관리감독자 또는 그 밖에 부서의 장(이하
“관리감독자등”이라 한다)에게
------------------------------------------------------------
[모델의 생각 / 답변]
산업안전보건법 **제6조**에 따른 근로자의 의무는 다음과 같습니다:

## 근로자의 기본 의무 (제6조)
1. **법규 준수**: 산업안전보건법과 이 법에 따른 명령으로 정하는 **산업재해 예방 기준을 지켜야 함**
2. **조치 이행**: 사업주, 노동감독관, 공단 등 관계인이 실시하는 **산업재해 예방 조치에 따라야 함**

## 추가 의무

**제40조(안전조치 및 보건조치 준수)**
- 사업주가 한 안전보건 조치를 지켜야 함

**제52조(작업중지)**
- 산업재해가 발생할 급박한 위험이 있는 경우 **작업을 중지하고 대피할 수 있으며**, 지체 없이 관리감독자에게 보고해야 함

즉, 근로자는 안전보건 관련 규정을 지키고 사업주의 안전 조치에 협조하며, 위급상황에서는 자신의 신체를 보호하기 위해 작업을 중지할 수 있습니다.
-------------------------------------------

#### 시나리오 B — 날씨 질문 → 검색 도구는 부르지 않는다

안전·법령과 무관한 날씨 질문입니다. `2.2` 의 고정 RAG라면 이런 질문에도 문서를 검색했겠지만, 이 에이전트는 **`search_osh_act` 를 부르지 않고** `get_today`·`get_weather` 만 골라 쓰는지 확인하세요.

In [10]:
_ = run_react(agent, "오늘 서울 날씨 어때?")

[사용자 질문]
오늘 서울 날씨 어때?
------------------------------------------------------------
[모델의 생각 / 답변]
오늘 서울 날씨를 확인해드리겠습니다.
[도구 호출] get_weather (인자: {'city': '서울'})
------------------------------------------------------------
[도구 결과] get_weather → 맑음 (최고 24도 / 최저 14도)
------------------------------------------------------------
[모델의 생각 / 답변]
오늘 서울 날씨는 **맑음**입니다. 최고 기온 24도, 최저 기온 14도로 예상됩니다. 쾌적한 날씨가 될 것 같으니 외출하기 좋은 날이네요!
------------------------------------------------------------


#### 시나리오 C — 복합 질문 → 여러 도구를 함께 사용

성격이 다른 두 가지를 한 번에 물어봅니다. 에이전트가 `get_today`(날짜)와 `search_osh_act`(법령)를 **모두** 골라 쓰며 답을 조립하는 과정을 살펴보세요.

In [11]:
_ = run_react(agent, "오늘 날짜 알려주고, 사업주의 안전·보건 조치 의무도 알려줘.")

[사용자 질문]
오늘 날짜 알려주고, 사업주의 안전·보건 조치 의무도 알려줘.
------------------------------------------------------------
[모델의 생각 / 답변]
오늘 날짜를 확인하고, 산업안전보건법에서 사업주의 안전·보건 조치 의무를 찾아드리겠습니다.
[도구 호출] get_today (인자: {})
[도구 호출] search_osh_act (인자: {'query': '사업주의 안전보건 조치 의무'})
------------------------------------------------------------
[도구 결과] get_today → 2026년 06월 23일
------------------------------------------------------------
[도구 결과] search_osh_act → (p.10) 법제처                                                            11                                                       국가법령정보센터
산업안전보건법
2. 방사선ㆍ유해광선ㆍ고열ㆍ한랭ㆍ초음파ㆍ소음ㆍ진동ㆍ이상기압 등에 의한 건강장해
3. 사업장에서 배출되
------------------------------------------------------------
[모델의 생각 / 답변]
## 📅 오늘 날짜
**2026년 06월 23일**

---

## 📋 사업주의 안전·보건 조치 의무

산업안전보건법 **제5조**에서 사업주의 주요 의무는 다음과 같습니다:

### 기본 의무
사업주는 다음 사항을 이행함으로써 근로자의 안전 및 건강을 유지·증진시켜야 합니다:

1. **산업재해 예방 기준 준수**
   - 법령으로 정하는 산업재해 예방을 위한 기준 준수

2. **쾌적한 작업환경 조성**
   - 신체적 피로와 정신적 스트레스를 줄일 수 있는 쾌적한 작업환경 조성
   - 근로조건 개선

3

#### 시나리오 D — 도구가 필요 없는 질문 → 바로 답변

인사나 자기소개 요청처럼 외부 정보가 필요 없는 질문입니다. 에이전트가 **도구를 한 번도 호출하지 않고** 바로 답하는지 확인합니다. (불필요한 검색을 하지 않는 것도 좋은 에이전트의 능력입니다.)

In [12]:
_ = run_react(agent, "안녕? 너는 무슨 일을 도와줄 수 있어?")

[사용자 질문]
안녕? 너는 무슨 일을 도와줄 수 있어?
------------------------------------------------------------
[모델의 생각 / 답변]
안녕하세요! 👋

저는 두 가지 분야에서 도움을 드릴 수 있습니다:

## 1️⃣ **산업안전보건법 관련 질문**
- 산업안전·보건 의무와 규정
- 사업주/근로자의 책임과 의무
- 중대재해, 도급, 작업중지 등의 법적 기준
- 안전보건진단 및 안전 관련 규정
- 기타 법령·안전 관련 사항

## 2️⃣ **생활 정보**
- **오늘 날짜** 확인
- **날씨 정보** (특정 도시의 오늘 날씨)

궁금한 것이 있으시면 편하게 물어보세요! 
- 산업안전보건법에 대해 알고 싶으신가요?
- 날씨를 확인하고 싶으신가요?
- 아니면 다른 정보가 필요하신가요?

무엇을 도와드릴까요? 😊
------------------------------------------------------------


#### 시나리오 E — 범위 밖 법령 질문 → 그라운딩으로 솔직하게

"연차 휴가" 는 근로기준법 영역이라 산업안전보건법 문서에는 없습니다. `2.2` 에서 봤듯, 에이전트가 검색해 보고 **근거가 없으면** 지어내지 말고 "찾을 수 없다"고 답해야 합니다. 그라운딩 규칙이 에이전트 맥락에서도 작동하는지 확인합니다.

In [13]:
_ = run_react(agent, "근로자의 연차 휴가는 며칠인가요?")

[사용자 질문]
근로자의 연차 휴가는 며칠인가요?
------------------------------------------------------------
[모델의 생각 / 답변]
근로자의 연차 휴가 관련 규정을 확인하기 위해 산업안전보건법을 검색하겠습니다.
[도구 호출] search_osh_act (인자: {'query': '근로자 연차 휴가'})
------------------------------------------------------------
[도구 결과] search_osh_act → (p.12) 작업장소에서 대피시키는 등 안전 및 보건에 관하여 필요한 조치를 하여야 한다.
 
제52조(근로자의 작업중지) ① 근로자는 산업재해가 발생할 급박한 위험이 있는 경우에는 작업을 중지하고 대피할 수
있다.
② 제1항에 따라 작업을 중지하고 대피한 근로자는 지체 없이 그 사실을 관리감독자 또는 그 밖에 부서의 장(이하
“관리감독자등”이라 한다)에게
------------------------------------------------------------
[모델의 생각 / 답변]
죄송합니다. 검색 결과에서 연차 휴가 관련 내용을 찾을 수 없었습니다.

**연차 휴가는 산업안전보건법이 아닌 『근로기준법』에서 규정하는 사항입니다.**

산업안전보건법은 주로 산업재해 예방, 안전보건 조치 등을 다루고 있으며, 휴가 관련 규정은 근로기준법에 명시되어 있습니다.

근로기준법에 따르면 근로자는 **1년간 8할 이상 출근한 경우 15일의 유급휴가(연차 휴가)**를 받을 수 있습니다. 자세한 내용은 근로기준법을 확인하시기 바랍니다.

혹시 산업안전보건법 관련 다른 질문이 있으시면 도와드리겠습니다.
------------------------------------------------------------


### 7. 2.2 RAG vs 2.3 Agentic RAG

| 항목 | 2.2 고정 RAG | 2.3 Agentic RAG |
|------|--------------|------------------|
| 검색 시점 | **항상** (모든 질문) | **필요할 때만** (에이전트가 판단) |
| 라우팅 주체 | 사람이 짠 코드 흐름 | **에이전트(LLM)** 의 도구 선택 |
| 도구 확장 | 검색 외 기능 추가 어려움 | 도구를 목록에 추가만 하면 됨 (날짜·날씨 등) |
| 비용·지연 | 일정 (매번 1회 검색) | 가변 (도구를 0~여러 번 호출) |
| 적합한 곳 | 항상 문서 검색이 필요한 단일 목적 QA | 검색 + 다른 작업이 섞인 범용 어시스턴트 |

**범위 밖 질문(예: 연차 휴가)** 에서 차이가 잘 드러납니다.
- `2.2` : 일단 검색한 뒤, 그라운딩 규칙 덕분에 "문서에 없음" 으로 답한다. (검색은 *발생*)
- `2.3` : 에이전트가 검색해 보고 근거가 없으면 "없음" 으로 답하거나, 애초에 관련 없다고 판단할 수 있다. **도구 선택의 주도권이 에이전트에 있다.**

> 둘은 우열이 아니라 **용도가 다릅니다.** 단일 목적 문서 QA라면 2.2가 단순·예측 가능해 좋고, 검색과 다른 기능이 섞이면 2.3이 유연합니다.

### 정리

- **Agentic RAG** 는 검색(retriever)을 *항상 거치는 단계* 가 아니라 **에이전트가 골라 쓰는 도구** 로 바꾼 것이다.
- retriever를 `@tool` 로 감싸면(`search_osh_act`), 에이전트가 *법령 질문일 때만* 검색하고 다른 질문은 다른 도구로 처리한다.
- 라우팅 품질의 핵심은 **① 도구의 독스트링** (언제 쓰는 도구인지)과 **② 시스템 프롬프트** (도구 선택 규칙 + 그라운딩)이다.
- `2.2` 의 교훈인 **그라운딩**("문서에 없으면 모른다고 답하라")은 에이전트 맥락에서도 그대로 유효하다.

> **좋은 Agentic RAG = 좋은 도구 설명 + 좋은 시스템 프롬프트(라우팅 · 그라운딩).**